# 🧠 Understanding Cosine Similarity in RAG
### *A Beginner-Friendly, Hands-On Guide using SentenceTransformers & Scikit-Learn*

---

## 🎯 What You Will Learn in This Notebook
1. **What is a Vector?** — How words and sentences turn into coordinates.
2. **What is Cosine Similarity?** — Why we measure the *angle* between vectors.
3. **Hands-On with `sentence-transformers`** — Running a real AI embedding model locally for free.
4. **The "Apple" Disambiguation** — Seeing Self-Attention & Cosine Similarity differentiate an iPhone from fresh fruit.
5. **Why Length Doesn't Fool Cosine Similarity** — Short sentences vs. long paragraphs.
6. **Build Your First Mini-RAG Retriever** — Finding the right document chunk for any user question in 10 lines of code.

---
## Step 1: The Intuition — What is Cosine Similarity?

Imagine two arrows starting from the origin `(0, 0)` on a graph:
- If both arrows point in the **exact same direction**, the angle between them is **$0^\circ$**, and the **cosine is $1.0$** (identical meaning).
- If the arrows are **perpendicular** (right angle, $90^\circ$), the **cosine is $0.0$** (completely unrelated).
- If they point in **opposite directions** ($180^\circ$), the **cosine is $-1.0$**.

Let's see this in pure Python with simple 2D numbers first before using AI.

In [1]:
import math

def calculate_cosine_similarity(vector_a, vector_b):
    """
    Calculate cosine similarity from scratch:
    Formula: Dot Product / (Magnitude of A * Magnitude of B)
    """
    # 1. Multiply matching numbers and sum them up (Dot Product)
    dot_product = sum(a * b for a, b in zip(vector_a, vector_b))
    
    # 2. Calculate the length (magnitude) of each vector
    magnitude_a = math.sqrt(sum(a ** 2 for a in vector_a))
    magnitude_b = math.sqrt(sum(b ** 2 for b in vector_b))
    
    # 3. Divide dot product by product of magnitudes
    return dot_product / (magnitude_a * magnitude_b)

# Arrow A: [1, 2]
# Arrow B: [2, 4] -> Exactly same direction as A, just twice as long!
# Arrow C: [-2, 1] -> Perpendicular to A (90 degree angle)
vec_a = [1.0, 2.0]
vec_b = [2.0, 4.0]
vec_c = [-2.0, 1.0]

print(f"Similarity between A and B (same direction, different length): {calculate_cosine_similarity(vec_a, vec_b):.2f}")
print(f"Similarity between A and C (completely perpendicular):        {calculate_cosine_similarity(vec_a, vec_c):.2f}")

Similarity between A and B (same direction, different length): 1.00
Similarity between A and C (completely perpendicular):        0.00


> 💡 **Notice:** Even though Arrow B is longer than Arrow A, their similarity is **1.00**!  
> This is why text search loves Cosine Similarity: a 5-word sentence and a 50-word paragraph on the same topic will have a similarity close to 1.0.

---
## Step 2: Load the AI Model (`sentence-transformers`)

We use `all-MiniLM-L6-v2`:
- It's completely **free** and runs **100% locally** on your machine.
- It converts any English text into **384 numbers** (a 384-dimensional vector).
- It has learned the meaning of words and sentences from billions of texts.

In [2]:
from sentence_transformers import SentenceTransformer

print("Loading model 'all-MiniLM-L6-v2'...")
model = SentenceTransformer("all-MiniLM-L6-v2")
print("Model loaded successfully!")

/Users/anandhu/Documents/Bosscoder notes/module-3/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading model 'all-MiniLM-L6-v2'...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 15865.93it/s]


Model loaded successfully!


Let's see what a real embedding vector looks like:

In [3]:
sample_text = "I love learning AI and RAG"
vector = model.encode(sample_text)

print(f"Input Text: '{sample_text}'")
print(f"Vector shape: {vector.shape} (meaning it is a list of {len(vector)} numbers)")
print(f"First 5 numbers: {vector[:5]}")

Input Text: 'I love learning AI and RAG'
Vector shape: (384,) (meaning it is a list of 384 numbers)
First 5 numbers: [-0.03606322 -0.06011819  0.05116258 -0.03867038  0.01258921]


---
## Step 3: Comparing Words with `sklearn` Cosine Similarity

Instead of writing loops by hand, in production we use `scikit-learn` (`sklearn.metrics.pairwise.cosine_similarity`), which computes this in milliseconds.

In [4]:
from sklearn.metrics.pairwise import cosine_similarity

words = ["dog", "puppy", "france"]
word_vectors = model.encode(words)

# Calculate similarity between 'dog' and 'puppy', and 'dog' and 'france'
sim_dog_puppy = cosine_similarity([word_vectors[0]], [word_vectors[1]])[0][0]
sim_dog_france = cosine_similarity([word_vectors[0]], [word_vectors[2]])[0][0]

print(f"Cosine Similarity ('dog' vs 'puppy'):  {sim_dog_puppy:.4f}  <-- High (both are canines!)")
print(f"Cosine Similarity ('dog' vs 'france'): {sim_dog_france:.4f}  <-- Low (completely unrelated)")

Cosine Similarity ('dog' vs 'puppy'):  0.8040  <-- High (both are canines!)
Cosine Similarity ('dog' vs 'france'): 0.3334  <-- Low (completely unrelated)


---
## Step 4: The "Apple" Problem (Context & Self-Attention)

**The Problem:** The word "apple" can mean an electronic device or a fruit.  
**User Query:** `"the apple i received was not turning on"`  

Notice the phrase **"not turning on"**. Because modern embeddings use **Self-Attention**, the model knows that turning on/off relates to electronic devices, not fruit!

Let's see which document chunk gets a higher cosine similarity:

In [10]:
query = "the apple i received was not turning on"

documents = [
    "Apple iPhone support: what to do if your device does not turn on or the screen is black",
    "Fresh Shimla apples: sweet red organic fruits available in 1kg packs from the orchard"
]

# 1. Embed query and documents
query_vector = model.encode([query])
doc_vectors = model.encode(documents)

# 2. Calculate cosine similarity
scores = cosine_similarity(query_vector, doc_vectors)[0]

print("Score" , scores)

for doc, score in zip(documents, scores):
    print(f"Score: {score:.4f} | Document: {doc}")

if scores[0] > scores[1]:
    print("\n✅ The retriever matched the iPhone support page (Score: {0:.4f} vs {1:.4f}) because 'not turning on' implies electronics!".format(scores[0], scores[1]))
else:
    print("\n❌ Match failed.")

Score [0.5693433  0.23972028]
Score: 0.5693 | Document: Apple iPhone support: what to do if your device does not turn on or the screen is black
Score: 0.2397 | Document: Fresh Shimla apples: sweet red organic fruits available in 1kg packs from the orchard

✅ The retriever matched the iPhone support page (Score: 0.5693 vs 0.2397) because 'not turning on' implies electronics!


---
## Step 5: Short vs. Long Text (Why Angle Beats Keyword Search)

Let's compare:
1. A short, direct request: `"refund my order"`
2. A wordy, formal paragraph: `"Dear support, I am writing to politely ask for my money back because the item I received was damaged during shipping."`
3. An unrelated question: `"How do I update my shipping address?"`

In [6]:
short_text = "refund my order"
long_text = "Dear support, I am writing to politely ask for my money back because the item I received was damaged during shipping."
unrelated_text = "How do I update my shipping address?"

test_vectors = model.encode([short_text, long_text, unrelated_text])

sim_short_long = cosine_similarity([test_vectors[0]], [test_vectors[1]])[0][0]
sim_short_unrelated = cosine_similarity([test_vectors[0]], [test_vectors[2]])[0][0]

print(f"Similarity (Short 'refund' vs Long 'money back request'): {sim_short_long:.4f}  <-- Strong match!")
print(f"Similarity (Short 'refund' vs 'shipping address'):        {sim_short_unrelated:.4f}  <-- Much lower")

Similarity (Short 'refund' vs Long 'money back request'): 0.5059  <-- Strong match!
Similarity (Short 'refund' vs 'shipping address'):        0.3624  <-- Much lower


---
## Step 6: Build a Mini-RAG Retriever

Now let's see **why cosine similarity is the heart of RAG**!

In a real RAG system:
1. You store all company documents as vectors in a database.
2. A user asks a question.
3. You compute **cosine similarity** between the user's question and all documents.
4. You pick the document with the **highest cosine similarity score** and send it to the LLM.

In [11]:
import numpy as np

# 1. Our company knowledge base (represented as document chunks)
knowledge_base = [
    {
        "id": 1,
        "title": "Refund Policy",
        "content": "Customers are eligible for a 100% refund within 30 days of purchase upon returning the item."
    },
    {
        "id": 2,
        "title": "Delivery Times",
        "content": "Standard shipping takes 3-5 business days. Express shipping delivers within 24 hours."
    },
    {
        "id": 3,
        "title": "IT Device Support",
        "content": "If your company laptop fails to turn on or suffers hardware damage, contact IT support on Slack at #it-help."
    },
    {
        "id": 4,
        "title": "Leave Policy",
        "content": "Full-time employees receive 18 paid vacation days and 12 sick leaves annually. Apply via HR portal."
    }
]

# 2. Pre-embed all documents (this is done once during ingestion)
doc_texts = [doc["content"] for doc in knowledge_base]
rag_doc_vectors = model.encode(doc_texts)
print(f"Indexed {len(knowledge_base)} documents into vectors!\n")

# 3. The RAG Retriever Function
def rag_retriever(user_question, top_k=1):
    # Embed the user question
    question_vec = model.encode([user_question])
    
    # Calculate cosine similarity against all stored documents
    scores = cosine_similarity(question_vec, rag_doc_vectors)[0]
    
    # Find the best matches (highest scores)
    best_indices = np.argsort(scores)[::-1][:top_k]
    
    print(f"❓ Question: '{user_question}'")
    print("=" * 60)
    for rank, idx in enumerate(best_indices, start=1):
        print(f"Top {rank} Match (Similarity Score: {scores[idx]:.4f})")
        print(f"📄 Document: {knowledge_base[idx]['title']}")
        print(f"💬 Content:  {knowledge_base[idx]['content']}\n")

Indexed 4 documents into vectors!



Let's test our retriever with different user questions!  
Notice that the questions don't use the same words as the documents, but the retriever finds the right chunk every single time:

In [12]:
# Test 1: Asking about money without using the word 'refund'
rag_retriever("Can I get my money back if I do not like the product?")

# Test 2: Asking about hardware issues
rag_retriever("My MacBook screen is black and will not boot up")

# Test 3: Asking about holidays
rag_retriever("How many days can I take off for vacation?")

❓ Question: 'Can I get my money back if I do not like the product?'
Top 1 Match (Similarity Score: 0.5247)
📄 Document: Refund Policy
💬 Content:  Customers are eligible for a 100% refund within 30 days of purchase upon returning the item.

❓ Question: 'My MacBook screen is black and will not boot up'
Top 1 Match (Similarity Score: 0.3090)
📄 Document: IT Device Support
💬 Content:  If your company laptop fails to turn on or suffers hardware damage, contact IT support on Slack at #it-help.

❓ Question: 'How many days can I take off for vacation?'
Top 1 Match (Similarity Score: 0.5701)
📄 Document: Leave Policy
💬 Content:  Full-time employees receive 18 paid vacation days and 12 sick leaves annually. Apply via HR portal.



---
## 🧪 Try Your Own Question!
Change the text in the cell below and run it to see which document chunk gets retrieved:

In [14]:
my_custom_question = "Who is president of america?"

rag_retriever(my_custom_question, top_k=2)

❓ Question: 'Who is president of america?'
Top 1 Match (Similarity Score: -0.0260)
📄 Document: Leave Policy
💬 Content:  Full-time employees receive 18 paid vacation days and 12 sick leaves annually. Apply via HR portal.

Top 2 Match (Similarity Score: -0.0469)
📄 Document: Delivery Times
💬 Content:  Standard shipping takes 3-5 business days. Express shipping delivers within 24 hours.



---
## 🎓 Summary: Why Cosine Similarity is Essential for RAG

1. **Semantic Understanding:** It measures *concept direction*, finding answers even when the user and the author use completely different words.
2. **Length-Independent:** Long documents and short questions are compared fairly based on topic rather than word count.
3. **Simple & Fast:** Calculating cosine similarity takes fractions of a millisecond, making it scalable to search millions of document chunks.
4. **The Foundation of Vector Databases:** Every vector database (ChromaDB, Pinecone, FAISS, Milvus) uses this exact math under the hood!